# 06 — Multi-task FinBERT (sentiment + controversy + movement)

Fine-tunes `ProsusAI/finbert` with three heads on a shared encoder, as described in `docs/NewsLens.md` §5.3. Labels come from `05_labels.ipynb` (`data/labels/multitask_labels.parquet`).

| Head | Task | Loss | Metric |
|---|---|---|---|
| 1 | Sentiment regression, `tanh` in [-1, 1] | MSE | Pearson r, MSE |
| 2 | Controversy binary classifier | BCE (masked) | F1, ROC-AUC |
| 3 | Future movement binary classifier | BCE (masked) | F1, ROC-AUC |

Total loss: `λ₁·MSE + λ₂·BCE_contr + λ₃·BCE_mov` with per-sample masks zeroing out unlabeled rows.

### Runtime modes
- `DRY_RUN = True` → 5k train / 1k val articles, 1 epoch. A 10-min sanity check for the full loop.
- `DRY_RUN = False` → full training on all 89k train articles, 3 epochs.

### Colab / local
The notebook auto-detects CUDA / MPS / CPU. When running in Colab, upload `data/labels/multitask_labels.parquet` (~50 MB) and set `DATA_DIR` below to the upload path, or mount Google Drive.

## 1 — Config

In [3]:
from pathlib import Path

# ── Runtime ────────────────────────────────────────────────────────────
DRY_RUN      = False   # flip to False for the real run
SEED         = 42

# ── Paths ──────────────────────────────────────────────────────────────
# For Colab: after uploading the parquet, set this to its directory.
# e.g. DATA_DIR = Path('/content')  and LABELS_PATH = DATA_DIR / 'multitask_labels.parquet'
DATA_DIR     = Path('../data')
LABELS_PATH  = DATA_DIR / 'labels' / 'multitask_labels.parquet'
MODEL_DIR    = Path('../models/finbert_multitask')
PRED_PATH    = DATA_DIR / 'predictions' / 'multitask_predictions.parquet'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Model / training hyperparameters ───────────────────────────────────
BASE_MODEL    = 'ProsusAI/finbert'
MAX_LEN       = 256           # FinBERT supports 512; 256 fits Lsa_summary comfortably and is 2× faster
BATCH_SIZE    = 16
EPOCHS        = 1 if DRY_RUN else 3
LR_ENCODER    = 2e-5
LR_HEADS      = 1e-4
WEIGHT_DECAY  = 0.01
WARMUP_FRAC   = 0.1

# Loss weights — sentiment is the anchor task; the two BCE heads are half-weighted
# so they don't dominate once their losses start dropping fast.
LAMBDA_SENT   = 1.0
LAMBDA_CONTR  = 0.5
LAMBDA_MOV    = 0.5

DRY_TRAIN_N   = 5000
DRY_VAL_N     = 1000

In [4]:
import os, random, json, math
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, roc_auc_score
from scipy.stats import pearsonr
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('device:', DEVICE)

device: mps


## 1b — Colab setup (skip when running locally)

Run this cell only on Colab. It installs extra packages not in Colab's default image, prompts you to upload `multitask_labels.parquet` from your machine, and rewrites the paths from cell 1 to point at `/content`. When running locally in VS Code with `.venv`, **skip this cell** — cell 1's paths are already correct.

In [4]:
# Install deps that aren't in Colab's default image
!pip install -q polars

# Upload multitask_labels.parquet from your local machine
from google.colab import files
uploaded = files.upload()

# Rewrite paths to Colab's /content filesystem
from pathlib import Path
DATA_DIR    = Path('/content')
LABELS_PATH = DATA_DIR / 'multitask_labels.parquet'
MODEL_DIR   = Path('/content/models/finbert_multitask')
PRED_PATH   = Path('/content/predictions/multitask_predictions.parquet')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)
print('LABELS_PATH exists:', LABELS_PATH.exists())

KeyboardInterrupt: 

## 2 — Load labels and build splits

In [7]:
labels = pl.read_parquet(LABELS_PATH)
print('rows:', labels.height)
print(labels.group_by('split').len().sort('split'))

train_df = labels.filter(pl.col('split') == 'train')
val_df   = labels.filter(pl.col('split') == 'val')
test_df  = labels.filter(pl.col('split') == 'test')

if DRY_RUN:
    train_df = train_df.sample(n=min(DRY_TRAIN_N, train_df.height), seed=SEED)
    val_df   = val_df.sample(n=min(DRY_VAL_N, val_df.height),       seed=SEED)
    print(f'DRY_RUN → train={train_df.height:,}  val={val_df.height:,}')

rows: 139522
shape: (3, 2)
┌───────┬───────┐
│ split ┆ len   │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ test  ┆ 28283 │
│ train ┆ 89549 │
│ val   ┆ 21690 │
└───────┴───────┘


## 3 — Dataset and DataLoader

The dataset yields tokenized text plus all three labels and both masks. We tokenize on the fly (fast enough for 90k rows with MiniLM-sized tokenizer) to keep memory flat.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class MultiTaskNewsDataset(Dataset):
    def __init__(self, df: pl.DataFrame, tokenizer, max_len: int):
        self.texts = df['text'].to_list()
        self.sent  = df['sentiment_label'].to_numpy().astype(np.float32)
        self.contr = df['controversy_label'].to_numpy().astype(np.float32)
        self.mov   = df['movement_label'].to_numpy().astype(np.float32)
        self.contr_mask = df['controversy_mask'].to_numpy().astype(np.float32)
        self.mov_mask   = df['movement_mask'].to_numpy().astype(np.float32)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt',
        )
        return {
            'input_ids':        enc['input_ids'].squeeze(0),
            'attention_mask':   enc['attention_mask'].squeeze(0),
            'sentiment':        torch.tensor(self.sent[i]),
            'controversy':      torch.tensor(self.contr[i]),
            'movement':         torch.tensor(self.mov[i]),
            'controversy_mask': torch.tensor(self.contr_mask[i]),
            'movement_mask':    torch.tensor(self.mov_mask[i]),
        }

train_ds = MultiTaskNewsDataset(train_df, tokenizer, MAX_LEN)
val_ds   = MultiTaskNewsDataset(val_df,   tokenizer, MAX_LEN)

# num_workers=0 on macOS: multiprocessing workers can't pickle classes
# defined in notebook cells. pin_memory=False: not supported on MPS.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'train batches: {len(train_loader)},  val batches: {len(val_loader)}')

train batches: 5597,  val batches: 1356


## 4 — Model

Shared FinBERT encoder + three linear heads on the pooled `[CLS]` token. Sentiment uses `tanh` to constrain outputs to [-1, 1] matching the label range; the classifier heads output raw logits (BCE-with-logits handles the sigmoid internally).

In [9]:
class FinBertMultiTask(nn.Module):
    def __init__(self, base_model: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.sent_head  = nn.Linear(hidden, 1)
        self.contr_head = nn.Linear(hidden, 1)
        self.mov_head   = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]          # [CLS]
        h = self.dropout(pooled)
        return {
            'sentiment':   torch.tanh(self.sent_head(h)).squeeze(-1),
            'contr_logit': self.contr_head(h).squeeze(-1),
            'mov_logit':   self.mov_head(h).squeeze(-1),
        }

model = FinBertMultiTask(BASE_MODEL).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable parameters: {n_params/1e6:.1f}M')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable parameters: 109.5M


## 5 — Optimizer, scheduler, masked multi-task loss

The encoder gets a lower learning rate (2e-5) than the freshly-initialized heads (1e-4). The controversy and movement losses are computed per-sample, multiplied by their masks, and averaged **over labeled samples only** — so unlabeled rows contribute exactly zero gradient to those heads while still training the encoder via the sentiment loss.

In [11]:
encoder_params = list(model.encoder.parameters())
head_params    = list(model.sent_head.parameters()) \
               + list(model.contr_head.parameters()) \
               + list(model.mov_head.parameters())

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': LR_ENCODER},
    {'params': head_params,    'lr': LR_HEADS},
], weight_decay=WEIGHT_DECAY)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

mse_none = nn.MSELoss(reduction='none')
bce_none = nn.BCEWithLogitsLoss(reduction='none')

def multitask_loss(out, batch):
    # Sentiment: fully labeled, simple mean MSE
    l_sent = mse_none(out['sentiment'], batch['sentiment']).mean()

    # Controversy: masked mean (protect against all-masked batches)
    cm = batch['controversy_mask']
    l_contr_raw = bce_none(out['contr_logit'], batch['controversy']) * cm
    l_contr = l_contr_raw.sum() / cm.sum().clamp(min=1.0)

    # Movement: masked mean
    mm = batch['movement_mask']
    l_mov_raw = bce_none(out['mov_logit'], batch['movement']) * mm
    l_mov = l_mov_raw.sum() / mm.sum().clamp(min=1.0)

    total = LAMBDA_SENT * l_sent + LAMBDA_CONTR * l_contr + LAMBDA_MOV * l_mov
    return total, {'sent': l_sent.item(), 'contr': l_contr.item(), 'mov': l_mov.item()}

In [12]:
def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sent_pred, sent_true = [], []
    contr_pred, contr_true, contr_mask = [], [], []
    mov_pred,  mov_true,  mov_mask  = [], [], []
    losses = {'sent': 0.0, 'contr': 0.0, 'mov': 0.0, 'total': 0.0}
    n_batches = 0
    for batch in tqdm(loader, desc='eval', leave=False):
        batch = to_device(batch)
        out = model(batch['input_ids'], batch['attention_mask'])
        total, parts = multitask_loss(out, batch)
        losses['total'] += total.item()
        for k, v in parts.items():
            losses[k] += v
        n_batches += 1

        sent_pred.append(out['sentiment'].cpu().numpy())
        sent_true.append(batch['sentiment'].cpu().numpy())
        contr_pred.append(torch.sigmoid(out['contr_logit']).cpu().numpy())
        contr_true.append(batch['controversy'].cpu().numpy())
        contr_mask.append(batch['controversy_mask'].cpu().numpy())
        mov_pred.append(torch.sigmoid(out['mov_logit']).cpu().numpy())
        mov_true.append(batch['movement'].cpu().numpy())
        mov_mask.append(batch['movement_mask'].cpu().numpy())

    for k in losses: losses[k] /= max(n_batches, 1)
    sent_pred = np.concatenate(sent_pred); sent_true = np.concatenate(sent_true)
    contr_pred = np.concatenate(contr_pred); contr_true = np.concatenate(contr_true); contr_mask = np.concatenate(contr_mask).astype(bool)
    mov_pred = np.concatenate(mov_pred); mov_true = np.concatenate(mov_true); mov_mask = np.concatenate(mov_mask).astype(bool)

    metrics = {
        'loss_total': losses['total'],
        'loss_sent':  losses['sent'],
        'loss_contr': losses['contr'],
        'loss_mov':   losses['mov'],
        'sent_mse':   float(np.mean((sent_pred - sent_true) ** 2)),
        'sent_pearson': float(pearsonr(sent_pred, sent_true)[0]) if len(sent_pred) > 1 else float('nan'),
    }
    if contr_mask.any():
        metrics['contr_f1']  = float(f1_score(contr_true[contr_mask], (contr_pred[contr_mask] > 0.5).astype(int)))
        metrics['contr_auc'] = float(roc_auc_score(contr_true[contr_mask], contr_pred[contr_mask])) if len(np.unique(contr_true[contr_mask])) > 1 else float('nan')
    if mov_mask.any():
        metrics['mov_f1']  = float(f1_score(mov_true[mov_mask], (mov_pred[mov_mask] > 0.5).astype(int)))
        metrics['mov_auc'] = float(roc_auc_score(mov_true[mov_mask], mov_pred[mov_mask])) if len(np.unique(mov_true[mov_mask])) > 1 else float('nan')
    return metrics

In [ ]:
history = []
best_val = math.inf
use_amp = (DEVICE.type == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = {'total': 0.0, 'sent': 0.0, 'contr': 0.0, 'mov': 0.0}
    n = 0
    pbar = tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        batch = to_device(batch)
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(batch['input_ids'], batch['attention_mask'])
                total, parts = multitask_loss(out, batch)
            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(batch['input_ids'], batch['attention_mask'])
            total, parts = multitask_loss(out, batch)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        running['total'] += total.item()
        for k, v in parts.items():
            running[k] += v
        n += 1
        pbar.set_postfix(loss=f'{running["total"]/n:.4f}')

    train_metrics = {f'train_{k}': v / n for k, v in running.items()}
    val_metrics   = evaluate(model, val_loader)
    row = {'epoch': epoch, **train_metrics, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    print(json.dumps(row, indent=2, default=float))

    if val_metrics['loss_total'] < best_val:
        best_val = val_metrics['loss_total']
        ckpt_path = MODEL_DIR / 'best.pt'
        torch.save({
            'model_state': model.state_dict(),
            'epoch': epoch,
            'val_metrics': val_metrics,
            'config': {
                'base_model': BASE_MODEL, 'max_len': MAX_LEN,
                'lambdas': [LAMBDA_SENT, LAMBDA_CONTR, LAMBDA_MOV],
            },
        }, ckpt_path)
        print(f'  ↑ new best, saved to {ckpt_path}')

with open(MODEL_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2, default=float)


## 7 — Inference over all articles

Load the best checkpoint and run it over every article in the labels parquet (train + val + test). This produces the per-article `sentiment_score`, `controversy_prob`, `movement_prob` that downstream components (Page 1 scatter plot, Page 2 node encoding, SHAP explainer) will query.

In [13]:
ckpt = torch.load(MODEL_DIR / 'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

all_df = labels if not DRY_RUN else labels.sample(n=min(5000, labels.height), seed=SEED)
all_ds = MultiTaskNewsDataset(all_df, tokenizer, MAX_LEN)
all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=False)

s_all, c_all, m_all = [], [], []
with torch.no_grad():
    for batch in tqdm(all_loader, desc='inference'):
        batch = to_device(batch)
        out = model(batch['input_ids'], batch['attention_mask'])
        s_all.append(out['sentiment'].cpu().numpy())
        c_all.append(torch.sigmoid(out['contr_logit']).cpu().numpy())
        m_all.append(torch.sigmoid(out['mov_logit']).cpu().numpy())

preds = all_df.select(['article_id', 'ticker', 'pub_date']).with_columns([
    pl.Series('sentiment_score',  np.concatenate(s_all).astype(np.float32)),
    pl.Series('controversy_prob', np.concatenate(c_all).astype(np.float32)),
    pl.Series('movement_prob',    np.concatenate(m_all).astype(np.float32)),
])
preds.write_parquet(PRED_PATH)
print(f'wrote {PRED_PATH}  ({preds.height:,} rows)')
preds.head(5)

inference:   0%|          | 0/4361 [00:00<?, ?it/s]

KeyboardInterrupt: 